# 02 · Batch, Sequence, and Tensor Shapes

**Goal:** understand what `(B, T)` means, why we batch, how padding and masks work, and how decoder inputs/targets are built.

Concepts: *sequence length `T`*, *batch size `B`*, *padding*, *pad mask*, *step / epoch*, *teacher forcing shift*.

Standalone — needs `numpy` and `matplotlib`.

## Words you'll meet
- **Token / token id:** a small piece of text turned into a number, like a locker number that stands for one word or digit.
- **Sequence (length `T`):** an ordered list of tokens, like one sentence; `T` is how many items it has.
- **Batch (size `B`):** a stack of sequences handled together, like marking a whole pile of exam papers in one go instead of one by one.
- **Shape `(B, T)`:** the size of the grid, like a class register: `B` rows (examples) by `T` columns (positions).
- **Padding and mask:** padding is blank filler to make short rows as long as the longest; the mask is the tick sheet saying which slots are real.
- **Embedding / vector:** a list of numbers describing one token, like a record card or GPS coordinates; `d_model` is how many numbers are on each card.
- **Step and epoch:** a step is one round of learning on one batch; an epoch is one complete pass through all the data, like finishing one whole textbook.
- **Teacher forcing:** while practising, we show the model the correct previous answers, like a teacher handing you the work so far before you write the next line.

## 1. Sequence
A **sequence** is an ordered list of token ids. Its length is `T` (sometimes `seq_len`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PAD, SOS, EOS = 10, 11, 12      # same convention as this repo's sort task; digits are 0..9
seq = [3, 1, 4, 1, 5]
print("one sequence:", seq, "  T =", len(seq))

> 🧒 **In plain words:**
> The first line (`import numpy as np`) borrows a toolbox for working with grids of numbers; `matplotlib` is the toolbox for drawing charts. `PAD, SOS, EOS = 10, 11, 12` gives three special "tokens" their own ids: the digits already use 0 to 9, so the specials take the next numbers. Think of them as special stamps: blank filler, start, end.
> `len(seq)` just counts the items in the list.
> **Notice:** the output says `T = 5`. A sequence is only a list of numbers, and T is how long it is.
> **Try it:** add another number to `seq` and see T grow.

## 2. Batch
Training on one sequence at a time is slow and noisy. We stack `B` sequences into a 2-D array of shape **`(B, T)`**:

- axis 0 → which example (batch)
- axis 1 → which position in the sequence

GPUs/BLAS process the whole batch in one matrix operation.

In [ ]:
batch = np.array([
    [3, 1, 4, 1, 5],
    [9, 2, 6, 5, 3],
    [2, 7, 1, 8, 2],
    [0, 0, 7, 4, 4],
])
B, T = batch.shape
print("shape (B, T) =", batch.shape)
print(batch)
print("\nexample 2, position 3 ->", batch[2, 3])

> 🧒 **In plain words:**
> We stacked 4 lists into one grid, like a class register: each row is one example, each column is one position. `batch.shape` gives the grid size as (rows, columns), and we save those two numbers as `B` and `T`.
> `batch[2, 3]` means "go to row 2, column 3". Counting starts at 0, so row 2 is the third row, `[2 7 1 8 2]`, and column 3 is its fourth item.
> **Notice:** the shape printed is `(4, 5)`: 4 examples by 5 positions, and the answer is 8.
> **Try it:** print `batch[0, 0]`. You should get 3, the very first number in the grid.

## 3. Sequences of different lengths → padding
Arrays must be rectangular. Shorter sequences are filled with `<PAD>` up to the longest in the batch.

In [ ]:
sentences = [[5, 2, 7], [4, 4, 1, 9, 0, 3], [8], [6, 6]]

def pad_batch(seqs, pad_id=PAD):
    T = max(len(s) for s in seqs)
    return np.array([s + [pad_id] * (T - len(s)) for s in seqs])

padded = pad_batch(sentences)
print(padded)

> 🧒 **In plain words:**
> Our four sentences have lengths 3, 6, 1 and 2, but a grid must be a neat rectangle. So `pad_batch` first finds the longest length (`max(...)`, which is 6). Then for each sentence it adds `[pad_id] * (T - len(s))`, which means "repeat the filler id enough times to reach the longest length". It is like writing answers on lined paper and drawing dashes to fill the blank space at the end of short lines.
> **Notice:** the filler is the number 10 (our `PAD`). Every row now has 6 items.
> **Try it:** add a longer sentence such as `[1, 2, 3, 4, 5, 6, 7]` to `sentences` and see all rows grow to 7.

## 4. Pad mask
Pad positions carry no information; the model must **ignore** them (e.g. attention must not look at them, loss must not count them).
A boolean **mask** records which positions are real.

In [ ]:
mask = padded != PAD          # True = real token
print(mask.astype(int))
print("\nreal tokens :", mask.sum(), " / total slots:", mask.size)
print(f"wasted compute on padding: {100*(1-mask.mean()):.0f}%")

> 🧒 **In plain words:**
> `padded != PAD` asks about every slot in the grid: "is this NOT the filler?" The answer for each slot is True or False, so `mask` is a grid of yes/no answers, like a register ticking who is really present. `.astype(int)` turns True/False into 1/0 so it is easier to read.
> `mask.sum()` adds up the ones (counting real tokens), `mask.size` is the number of slots in the grid, and `mask.mean()` is the fraction that are real.
> **Notice:** 12 of 24 slots are real, so exactly half the grid, 50 percent, is filler. The computer still does work on those slots, which is wasted effort.
> **Try it:** change the sentences so they are all the same length. The waste should drop to 0 percent.

In [ ]:
plt.figure(figsize=(4, 2.2))
plt.imshow(mask, cmap="Greens", aspect="auto")
plt.xlabel("position t"); plt.ylabel("example b"); plt.title("pad mask (green = real)")
plt.show()

> 🧒 **In plain words:**
> `imshow` paints the grid of 1s and 0s as coloured squares: green for real tokens, white for filler. The other lines just add labels and a title, and `plt.show()` displays the picture.
> **Notice:** the picture is the same grid as the printout above. Row 1 (the second row) is fully green because that sentence is the longest; row 2 has just one green square.
> **Try it:** change `"Greens"` to `"Blues"` to recolour it.

Tip: batching sequences of **similar length together** (bucketing) reduces padding waste.

## 5. Adding the feature axis: `(B, T)` → `(B, T, d_model)`
After the embedding layer (next notebooks) each token id becomes a vector of length `d_model`.
Everything inside a Transformer layer is shaped **`(B, T, d_model)`**.

In [ ]:
d_model = 8
V = 13
rng = np.random.default_rng(0)
E = rng.normal(size=(V, d_model))        # embedding table (V, d_model)

x = E[padded]                            # fancy indexing: ids -> vectors
print("ids   :", padded.shape, " = (B, T)")
print("embeds:", x.shape,      " = (B, T, d_model)")

> 🧒 **In plain words:**
> This is the trickiest line so far, so let us go slowly.
> - `E` is a big table, like a library catalogue or a set of record cards. It has `V = 13` rows (one row for every token id, 0 to 12) and `d_model = 8` columns, so each row is a list of 8 numbers. `rng.normal(size=(V, d_model))` fills the table with random numbers.
> - `padded` is our grid of ids, shape (B, T), meaning B rows (examples) by T columns (positions): here 4 by 6.
> - `E[padded]` is called fancy indexing. When you put a whole grid of whole numbers inside the square brackets, it means: "for each id in the grid, go and fetch that row of the table". It is like replacing every student's roll number with their full record card. Each single number becomes a list of 8 numbers.
> - So the grid of shape (4, 6) becomes a block of shape (4, 6, 8): 4 examples by 6 positions by 8 numbers per position. That is why the output says `(4, 6, 8)`.
> - The padding id 10 also gets fetched, since it is just another row in the table. The model does not know that 10 means "filler"; that is exactly why we need the mask from section 4.
> - `E` is random here, so these vectors (lists of numbers) mean nothing yet. Notebook 03 explains how they become meaningful during training.
> **Notice:** the printout shows `(4, 6)` before and `(4, 6, 8)` after. Only a new last axis was added.
> **Try it:** print `x[0, 0]` to see 8 numbers, and `E[5]` to see the row for id 5. They are identical because `padded[0, 0]` is 5.

## 6. Batch size, steps, epochs
- **Step** (iteration): one forward + backward + parameter update on one batch.
- **Epoch**: one full pass over the dataset.

`steps_per_epoch = ceil(N / B)`

In [ ]:
N = 10_000
for B_ in [16, 64, 256, 1024]:
    print(f"B={B_:5d}  steps/epoch = {int(np.ceil(N / B_))}")

> 🧒 **In plain words:**
> We have N = 10,000 examples. The `for` loop tries four batch sizes one after another (the variable is called `B_` to avoid clobbering our earlier `B`). For each, `N / B_` says how many batches fit, and `np.ceil` rounds up, because a leftover partial batch still counts as a step. Think of dealing cards: more cards per hand means fewer hands.
> **Notice:** bigger batches mean fewer steps per epoch: 625 steps at B=16 but only 10 at B=1024.
> **Try it:** change N to 100 and see how the numbers shrink.

## 7. Why batch size matters: gradient noise
The gradient of a batch is the **average** of per-example gradients. A larger batch gives a less noisy estimate (std shrinks like `1/√B`), but each step costs more.

Simulation: pretend each example's gradient is `true_grad + noise`.

In [ ]:
rng = np.random.default_rng(1)
true_grad, noise_std = 1.0, 3.0
sizes = [1, 4, 16, 64, 256]
stds = []
for B_ in sizes:
    est = rng.normal(true_grad, noise_std, size=(5000, B_)).mean(axis=1)
    stds.append(est.std())

plt.figure(figsize=(5, 3))
plt.loglog(sizes, stds, "o-", label="measured")
plt.loglog(sizes, [noise_std / np.sqrt(b) for b in sizes], "--", label="noise/√B")
plt.xlabel("batch size B"); plt.ylabel("std of gradient estimate"); plt.legend(); plt.show()

> 🧒 **In plain words:**
> We pretend the true answer is 1.0, but every single example gives a noisy guess (the real value plus random wobble of size 3). For each batch size we make 5000 pretend batches, shape (5000, B_): 5000 rows, each with B_ guesses. `.mean(axis=1)` averages each row, so we get 5000 batch estimates, and `.std()` measures how much they wobble.
> The plot uses log scales on both axes so a power law looks like a straight line. The dashed line is the formula noise divided by the square root of B.
> **Notice:** the dots follow the dashed line closely: averaging more examples gives steadier estimates, like asking more classmates and averaging their opinions. But to halve the wobble you need 4 times as many examples.
> **Try it:** change `noise_std` to 6 and the whole line should shift up.

## 8. Encoder input, decoder input, decoder target (teacher forcing)
For a sequence-to-sequence task (here: sort digits) the model has **two** sequences:

- encoder input: the unsorted digits
- decoder input: `<SOS>` + the correct output (**shifted right**)
- decoder target: the correct output + `<EOS>`

At position `t` the decoder sees the answer up to `t-1` and must predict token `t`. Feeding the true previous tokens during training (instead of the model's own guesses) is called **teacher forcing**.

In [ ]:
enc_in = np.array([[3, 1, 4, 1, 5],
                   [9, 2, 6, 5, 3]])
sorted_ = np.sort(enc_in, axis=1)

sos = np.full((enc_in.shape[0], 1), SOS)
eos = np.full((enc_in.shape[0], 1), EOS)
dec_in  = np.concatenate([sos, sorted_], axis=1)
dec_tgt = np.concatenate([sorted_, eos], axis=1)

for b in range(2):
    print("encoder input :", enc_in[b])
    print("decoder input :", dec_in[b])
    print("decoder target:", dec_tgt[b])
    print()

> 🧒 **In plain words:**
> The task is sorting digits. `np.sort(enc_in, axis=1)` sorts each row on its own (axis=1 means "along the row"). `np.full((2, 1), SOS)` makes a column of two start markers, and the same for `EOS`, the end marker.
> `np.concatenate([...], axis=1)` glues columns side by side. The decoder input is the start marker followed by the sorted answer; the decoder target is the sorted answer followed by the end marker. So the input is the answer shifted one step to the right.
> **Notice:** the first example prints input `[11 1 1 3 4 5]` and target `[1 1 3 4 5 12]`. Compare them column by column: each target is the very next item after the input. The input lags one step behind, like a student reading the answer line one word at a time and guessing the next word.
> **Try it:** change `sorted_` to `enc_in[:, ::-1]` (reversed rows) to build a reverse task instead.

Read column by column: given `dec_in[t]` (and everything before it), predict `dec_tgt[t]`.

In [ ]:
b = 0
for t in range(dec_in.shape[1]):
    print(f"t={t}: sees {list(dec_in[b, :t+1])}  -> must predict {dec_tgt[b, t]}")

> 🧒 **In plain words:**
> For the first example we loop over every position `t`. `dec_in[b, :t+1]` means "from row b, take everything up to and including position t", which is what the decoder is allowed to see at that moment. The f-string then prints it next to the correct answer `dec_tgt[b, t]`.
> **Notice:** the list of seen tokens grows by one at every line, and the thing to predict is the next sorted digit; the last one is 12, the end marker. (The odd `np.int64(11)` wrapping is just how numpy shows its numbers inside a list; read it as plain 11.)
> **Try it:** set `b = 1` to watch the same process for the second example.

## 9. The big picture: what is looped and what is not
Only **two** levels are real `for` loops during training: epochs, and steps (one step = one batch).
Everything inside a batch (all `B` samples, all `T` positions, all `d_model` numbers) is handled **at once** as one array.

```
epoch      one pass over all N examples
 └─ step   one batch of B examples -> one parameter update   (ceil(N/B) per epoch)
     └─ sample      one example (a row of the batch)       ┐ computed together
         └─ position   one token in the sequence           ┘ as one (B, T, d_model) array
             └─ embedding   a vector of d_model numbers
```

In [ ]:
N, B, T, d_model = 1000, 64, 5, 8
ids = np.random.default_rng(0).integers(0, 10, size=(N, T))     # a pretend dataset of N sequences
E = np.random.default_rng(1).normal(size=(13, d_model))          # embedding table

num_epochs = 2
step_count = 0
for epoch in range(num_epochs):                       # loop 1: epochs
    order = np.random.default_rng(epoch).permutation(N)   # shuffle each epoch
    for start in range(0, N, B):                      # loop 2: steps (one batch each)
        batch = ids[order[start:start + B]]           # shape (B, T); last batch may be smaller
        x = E[batch]                                  # shape (B, T, d_model): NO loop over samples/positions
        step_count += 1                               # (a real step would also compute loss and update weights here)
    print(f"epoch {epoch}: finished; last batch shape {batch.shape}, x shape {x.shape}")

print("total steps:", step_count, "  expected:", num_epochs * int(np.ceil(N / B)))

> 🧒 **In plain words:**
> There are only two `for` loops. The outer one counts **epochs** (laps through the whole dataset). The inner one walks through the data `B` examples at a time; each time round is one **step**.
> The line `E[batch]` handles all 64 examples and all 5 positions in one go; no loop is needed for them. That is why GPUs love batches.
> **Notice:** the last batch of each epoch is smaller (1000 is not a multiple of 64: 15 full batches use 960, leaving 40), and the total steps match `epochs × ceil(N/B)`.
> **Try it:** change `B` to 100 and see that the last batch becomes full-size and the step count drops to 20.

Loops that **do** exist deeper inside: the stack of `N` encoder/decoder layers (a small loop over layers), and **generation** at inference time (producing tokens one by one). During training, teacher forcing lets all positions be computed at once.

## Recap
| name | meaning | typical |
|---|---|---|
| `B` | sequences per batch | 16–4096 |
| `T` | tokens per sequence | 5–100k |
| `V` | vocab size | 13–100k |
| `d_model` | vector length per token | 64–12288 |

Token ids are `(B, T)`; activations are `(B, T, d_model)`; the final scores are `(B, T, V)`.

➡ Next: **03 · Embeddings** — what those `d_model`-length vectors *mean*.